# Notebook 01 â€” Data Preparation

**Purpose**: Parse JSONL dataset, derive binary and type labels, perform audit checks, generate group-safe train/val/test splits.

**Outputs**:
- `outputs/datasets/binary_dataset.csv`
- `outputs/datasets/type_dataset.csv`
- `outputs/splits/train_binary.csv`, `val_binary.csv`, `test_binary.csv`
- `outputs/splits/train_type.csv`, `val_type.csv`, `test_type.csv`
- `outputs/splits/split_metadata.json`

In [ ]:
import os; from pathlib import Path as _P; print("cwd:", _P.cwd()); print("files:", list(_P.cwd().iterdir())[:10])

import json, hashlib, random, os
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FILENAME = "sarcasm_pairs_step35_clean.jsonl"

# ── Step 1: locate or upload the data file ───────────────────────────────────
def _locate_file(filename):
    """Search common locations; return Path if found, else None."""
    candidates = []
    for root in [Path.cwd()] + list(Path.cwd().parents):
        for sub in [
            Path("data") / "processed" / filename,
            Path("data") / filename,
            Path(filename),
        ]:
            candidates.append(root / sub)
    for p in [
        Path("/content") / filename,
        Path("/mnt/data") / filename,
        Path("/content/drive/MyDrive") / filename,
    ]:
        candidates.append(p)
    _content = Path("/content")
    for p in (_content.rglob(filename) if _content.exists() else []):
        candidates.append(p)
    for p in candidates:
        if p.is_file():
            return p
    return None

DATA_FILE = _locate_file(FILENAME)

if DATA_FILE is None:
    # ── Auto-upload in Colab ─────────────────────────────────────────────────
    try:
        from google.colab import files as _colab_files
        print(f"{FILENAME!r} not found. Please upload it now:")
        _uploaded = _colab_files.upload()
        if not _uploaded:
            raise RuntimeError("No file uploaded. Please upload the JSONL file and re-run.")
        _fname = list(_uploaded.keys())[0]
        if not _fname.endswith(FILENAME.split('/')[-1]):
            raise ValueError(
                f"Wrong file uploaded: {_fname!r}. Expected {FILENAME!r}."
            )
        DATA_FILE = Path("/content") / FILENAME
        if Path(_fname) != DATA_FILE:
            import shutil
            shutil.move(_fname, str(DATA_FILE))
        print(f"Uploaded to {DATA_FILE}")
    except ImportError:
        # Not in Colab
        raise FileNotFoundError(
            f"Cannot find {FILENAME!r}.\n"
            f"cwd: {Path.cwd()}\n"
            "Place the file at: <project_root>/data/processed/" + FILENAME
        )

# ── Step 2: determine project root ──────────────────────────────────────────
def _find_root(data_file):
    """Walk up from data_file to find the project root (contains outputs/ or notebooks/)."""
    markers = [
        Path("outputs"),
        Path("notebooks"),
        Path("data"),
    ]
    for parent in [data_file.parent] + list(data_file.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    return data_file.parent  # fallback

ROOT = _find_root(DATA_FILE)

OUT_DATASETS = ROOT / "outputs" / "datasets"
OUT_SPLITS   = ROOT / "outputs" / "splits"
OUT_DATASETS.mkdir(parents=True, exist_ok=True)
OUT_SPLITS.mkdir(parents=True, exist_ok=True)

print(f"Dataset : {DATA_FILE}")
print(f"Root    : {ROOT}")
print(f"Outputs : {OUT_DATASETS}")


## 1. Load and Validate Raw Data

In [ ]:
REQUIRED_FIELDS = {"original_headline", "generated_headline", "strategy", "type", "model_used", "article_link"}
ALLOWED_TYPES   = {"sarcastic_to_non", "non_to_sarcastic"}
ALLOWED_STRATS  = {"irony", "overstatement", "rhetorical_question", "sarcasm", "satire", "understatement"}

raw_rows = []
errors   = []

with open(DATA_FILE, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append((line_num, f"JSON parse error: {e}"))
            continue

        # Schema check
        missing = REQUIRED_FIELDS - set(row.keys())
        if missing:
            errors.append((line_num, f"Missing fields: {missing}"))
            continue

        # Value checks
        if row["type"] not in ALLOWED_TYPES:
            errors.append((line_num, f"Unknown type: {row['type']!r}"))
            continue
        if row["strategy"] not in ALLOWED_STRATS:
            errors.append((line_num, f"Unknown strategy: {row['strategy']!r}"))
            continue

        row["_line_num"] = line_num
        raw_rows.append(row)

print(f"Loaded rows : {len(raw_rows):,}")
print(f"Error rows  : {len(errors)}")
if errors:
    for ln, msg in errors[:5]:
        print(f"  Line {ln}: {msg}")

## 2. Dataset Audit

In [ ]:
# â”€â”€ Type and Strategy distributions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
type_counts     = Counter(r["type"]     for r in raw_rows)
strategy_counts = Counter(r["strategy"] for r in raw_rows)
model_counts    = Counter(r["model_used"] for r in raw_rows)

print("=== Type distribution ===")
for k, v in type_counts.most_common():
    print(f"  {k}: {v:,} ({v/len(raw_rows)*100:.1f}%)")

print("\n=== Strategy distribution ===")
for k, v in strategy_counts.most_common():
    print(f"  {k}: {v:,} ({v/len(raw_rows)*100:.1f}%)")

print("\n=== Model distribution ===")
for k, v in model_counts.most_common():
    print(f"  {k}: {v:,}")

In [ ]:
# â”€â”€ Duplicate checks â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
orig_headlines  = [r["original_headline"]  for r in raw_rows]
gen_headlines   = [r["generated_headline"] for r in raw_rows]
article_links   = [r["article_link"]       for r in raw_rows]

dup_orig    = len(orig_headlines) - len(set(orig_headlines))
dup_gen     = len(gen_headlines)  - len(set(gen_headlines))
dup_article = len(article_links)  - len(set(article_links))

print(f"Duplicate original_headlines : {dup_orig}")
print(f"Duplicate generated_headlines: {dup_gen}")
print(f"Duplicate article_links      : {dup_article}")

In [ ]:
# â”€â”€ Null / empty checks â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
for field in ["original_headline", "generated_headline", "strategy", "type", "model_used", "article_link"]:
    null_count  = sum(1 for r in raw_rows if r.get(field) is None)
    empty_count = sum(1 for r in raw_rows if r.get(field) == "")
    print(f"{field:25s}: {null_count} nulls, {empty_count} empty strings")

In [ ]:
# â”€â”€ Strategy distribution plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Type dist
ax = axes[0]
labels, counts = zip(*sorted(type_counts.items(), key=lambda x: -x[1]))
ax.barh(labels, counts, color=["steelblue", "coral"])
ax.set_title("Row-level Type Distribution", fontsize=14)
ax.set_xlabel("Count")
for i, c in enumerate(counts):
    ax.text(c + 50, i, f"{c:,}", va="center")

# Strategy dist
ax = axes[1]
labels2, counts2 = zip(*sorted(strategy_counts.items(), key=lambda x: -x[1]))
ax.barh(labels2, counts2, color="steelblue")
ax.set_title("Strategy Distribution (Type Task Labels)", fontsize=14)
ax.set_xlabel("Count")
for i, c in enumerate(counts2):
    ax.text(c + 30, i, f"{c:,}", va="center")

plt.tight_layout()
plt.savefig(OUT_DATASETS / "distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/datasets/distributions.png")

## 3. Label Derivation and Dataset Expansion

In [ ]:
from __future__ import annotations
from urllib.parse import urlparse


def normalize_url(url: str) -> str:
    """Lowercase and strip scheme/trailing slash for stable group IDs."""
    url = url.strip().lower()
    parsed = urlparse(url)
    normalized = (parsed.netloc + parsed.path).rstrip("/")
    return normalized if normalized else url


def derive_labels(raw_rows):
    """
    Expand each JSONL row into 2 samples (sarcastic + non-sarcastic).

    Rules:
      sarcastic_to_non -> original=sarcastic(1), generated=non-sarcastic(0)
      non_to_sarcastic -> original=non-sarcastic(0), generated=sarcastic(1)

    Returns binary_df, type_df
    """
    binary_records = []
    sample_id = 0

    for pair_id, row in enumerate(raw_rows):
        group_id   = normalize_url(row["article_link"]) if row["article_link"] else str(pair_id)
        row_type   = row["type"]
        strategy   = row["strategy"]
        model_used = row["model_used"]
        article    = row["article_link"]

        if row_type == "sarcastic_to_non":
            orig_label, orig_strat = 1, strategy
            gen_label,  gen_strat  = 0, None
        else:
            orig_label, orig_strat = 0, None
            gen_label,  gen_strat  = 1, strategy

        binary_records.append({
            "sample_id"    : sample_id,
            "pair_id"      : pair_id,
            "group_id"     : group_id,
            "text"         : row["original_headline"],
            "binary_label" : orig_label,
            "is_generated" : 0,
            "strategy"     : orig_strat,
            "source_type"  : row_type,
            "article_link" : article,
            "model_used"   : model_used,
        })
        sample_id += 1

        binary_records.append({
            "sample_id"    : sample_id,
            "pair_id"      : pair_id,
            "group_id"     : group_id,
            "text"         : row["generated_headline"],
            "binary_label" : gen_label,
            "is_generated" : 1,
            "strategy"     : gen_strat,
            "source_type"  : row_type,
            "article_link" : article,
            "model_used"   : model_used,
        })
        sample_id += 1

    binary_df = pd.DataFrame(binary_records)

    # Validate counts
    counts = binary_df["binary_label"].value_counts().to_dict()
    n_expected = len(raw_rows)
    n0 = int(counts.get(0, 0))
    n1 = int(counts.get(1, 0))
    if n0 != n_expected or n1 != n_expected:
        raise ValueError(
            f"Label count mismatch!\n"
            f"  Expected {n_expected} per class\n"
            f"  Got: sarcastic(1)={n1}, non-sarcastic(0)={n0}"
        )

    # Type dataset: sarcastic only
    type_df = binary_df[binary_df["binary_label"] == 1].copy()
    type_df = type_df.rename(columns={"strategy": "type_label"})
    type_df = type_df[["sample_id", "pair_id", "group_id", "text",
                        "type_label", "is_generated", "source_type",
                        "article_link", "model_used"]]

    missing = type_df["type_label"].isna().sum()
    if missing > 0:
        raise ValueError(f"{missing} sarcastic samples have no type_label!")

    return binary_df, type_df


binary_df, type_df = derive_labels(raw_rows)

print(f"Binary dataset : {len(binary_df):,} samples")
print(f"  sarcastic (1): {(binary_df['binary_label']==1).sum():,}")
print(f"  non-sarc  (0): {(binary_df['binary_label']==0).sum():,}")
print()
print(f"Type dataset   : {len(type_df):,} samples")
print(type_df["type_label"].value_counts().to_string())


In [ ]:
# Save full datasets
binary_df.to_csv(OUT_DATASETS / "binary_dataset.csv", index=False)
type_df.to_csv(  OUT_DATASETS / "type_dataset.csv",   index=False)
print("Saved: outputs/datasets/binary_dataset.csv")
print("Saved: outputs/datasets/type_dataset.csv")

## 4. Group-Aware Train / Val / Test Split

In [ ]:
def group_aware_split(
    df: pd.DataFrame,
    group_col: str = "group_id",
    train_frac: float = 0.70,
    val_frac:   float = 0.15,
    seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split df into train/val/test at the group level.
    All rows sharing a group_id go to exactly one split.
    """
    groups = df[group_col].unique().tolist()
    rng = np.random.default_rng(seed)
    rng.shuffle(groups)

    n = len(groups)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)

    train_groups = set(groups[:n_train])
    val_groups   = set(groups[n_train : n_train + n_val])
    test_groups  = set(groups[n_train + n_val :])

    train_df = df[df[group_col].isin(train_groups)].copy()
    val_df   = df[df[group_col].isin(val_groups)].copy()
    test_df  = df[df[group_col].isin(test_groups)].copy()

    # Sanity: no overlap
    assert train_groups.isdisjoint(val_groups),  "Train/val group overlap!"
    assert train_groups.isdisjoint(test_groups), "Train/test group overlap!"
    assert val_groups.isdisjoint(test_groups),   "Val/test group overlap!"

    return train_df, val_df, test_df


# â”€â”€ Binary splits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_bin, val_bin, test_bin = group_aware_split(binary_df)

print("=== Binary splits ===")
for name, df in [("train", train_bin), ("val", val_bin), ("test", test_bin)]:
    dist = df["binary_label"].value_counts().to_dict()
    print(f"  {name:5s}: {len(df):,} samples | sarcastic={dist.get(1,0):,} non={dist.get(0,0):,}")

# â”€â”€ Type splits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_type, val_type, test_type = group_aware_split(type_df)

print("\n=== Type splits ===")
for name, df in [("train", train_type), ("val", val_type), ("test", test_type)]:
    print(f"  {name:5s}: {len(df):,} samples")
    dist = df["type_label"].value_counts()
    for label, cnt in dist.items():
        print(f"    {label}: {cnt:,}")

In [ ]:
# â”€â”€ Verify no pair crosses split boundary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# A pair_id should appear in at most ONE binary split
train_pairs = set(train_bin["pair_id"])
val_pairs   = set(val_bin["pair_id"])
test_pairs  = set(test_bin["pair_id"])

tv_overlap  = train_pairs & val_pairs
tt_overlap  = train_pairs & test_pairs
vt_overlap  = val_pairs   & test_pairs

print(f"Train/Val pair_id overlap  : {len(tv_overlap)} (must be 0)")
print(f"Train/Test pair_id overlap : {len(tt_overlap)} (must be 0)")
print(f"Val/Test pair_id overlap   : {len(vt_overlap)} (must be 0)")

assert len(tv_overlap) == 0 and len(tt_overlap) == 0 and len(vt_overlap) == 0, \
    "Leakage detected: pairs crossing split boundaries!"
print("\nLeakage check PASSED: No pair_id crosses split boundaries.")

In [ ]:
# â”€â”€ Save splits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
split_files = {
    "train_binary": train_bin,
    "val_binary"  : val_bin,
    "test_binary" : test_bin,
    "train_type"  : train_type,
    "val_type"    : val_type,
    "test_type"   : test_type,
}
for fname, df in split_files.items():
    path = OUT_SPLITS / f"{fname}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path.relative_to(ROOT)}")

# Metadata
import json as _json
metadata = {
    "seed": SEED,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,
    "group_col": "group_id",
    "total_pairs": len(raw_rows),
    "binary": {
        "train": len(train_bin), "val": len(val_bin), "test": len(test_bin)
    },
    "type": {
        "train": len(train_type), "val": len(val_type), "test": len(test_type)
    },
}
with open(OUT_SPLITS / "split_metadata.json", "w") as f:
    _json.dump(metadata, f, indent=2)
print("\nSaved: outputs/splits/split_metadata.json")

## 5. Split Distribution Visualization

In [ ]:
# Type label distribution per split
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
split_dfs = [("Train", train_type), ("Val", val_type), ("Test", test_type)]

for ax, (name, df) in zip(axes, split_dfs):
    dist = df["type_label"].value_counts()
    dist.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"{name} â€” Type Labels (n={len(df):,})", fontsize=13)
    ax.set_xlabel("Count")
    for i, (label, cnt) in enumerate(dist.items()):
        ax.text(cnt + 5, i, f"{cnt:,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_SPLITS / "type_split_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/splits/type_split_distribution.png")

In [ ]:
# Binary label distribution per split
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
split_dfs_bin = [("Train", train_bin), ("Val", val_bin), ("Test", test_bin)]

for ax, (name, df) in zip(axes, split_dfs_bin):
    dist = df["binary_label"].value_counts().sort_index()
    ax.bar(["Non-sarcastic (0)", "Sarcastic (1)"], dist.values, color=["coral", "steelblue"])
    ax.set_title(f"{name} â€” Binary Labels (n={len(df):,})", fontsize=12)
    ax.set_ylabel("Count")
    for i, v in enumerate(dist.values):
        ax.text(i, v + 20, f"{v:,}", ha="center")

plt.tight_layout()
plt.savefig(OUT_SPLITS / "binary_split_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/splits/binary_split_distribution.png")

In [ ]:
print("\n=== Data Preparation Complete ===")
print(f"Binary dataset : {len(binary_df):,} samples (balanced)")
print(f"Type dataset   : {len(type_df):,} samples (imbalanced, 6 classes)")
print(f"Splits saved to: outputs/splits/")
print(f"Datasets saved : outputs/datasets/")
print("\nReady for classical baseline training (Notebooks 02 and 03).")